# ISIC Task 2: Attribute Feature Detection - Data Exploration

This notebook explores the attribute detection dataset and converts it to a multilabel classification problem.

**Dataset:**
- Images: `datasets/ISIC2018_Task1-2_Training_Input/ISIC_{id}.jpg`
- Attributes: `datasets/ISIC2018_Task2_Training_GroundTruth_v3/ISIC_{id}_attribute_{type}.png`

**Five Attribute Types:**
1. globules
2. milia_like_cyst
3. negative_network
4. pigment_network
5. streaks

In [ ]:
import sys
sys.path.append('..')

import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from preprocessing.attr_preprocessing import (
    process_dataset_labels,
    get_label_statistics,
    ATTR_TYPES,
    FEAT_THRESHOLD
)

## 1. Dataset Paths

In [ ]:
# Define paths
IMAGE_FOLDER = '../datasets/ISIC2018_Task1-2_Training_Input'
GT_FOLDER = '../datasets/ISIC2018_Task2_Training_GroundTruth_v3'

print(f"Image folder exists: {os.path.exists(IMAGE_FOLDER)}")
print(f"GT folder exists: {os.path.exists(GT_FOLDER)}")
print(f"\nFeature threshold: {FEAT_THRESHOLD}")
print(f"Attribute types: {ATTR_TYPES}")

## 2. Process Dataset to Multilabel Format

In [ ]:
# Process all images and generate labels
print("Processing dataset...")
image_ids, labels_matrix = process_dataset_labels(IMAGE_FOLDER, GT_FOLDER)

print(f"\nProcessed {len(image_ids)} images")
print(f"Labels matrix shape: {labels_matrix.shape}")
print(f"\nFirst 5 image IDs: {image_ids[:5]}")
print(f"\nFirst 5 label vectors:\n{labels_matrix[:5]}")

## 3. Label Statistics

In [ ]:
# Compute statistics
stats = get_label_statistics(labels_matrix)

print("\nAttribute Statistics:")
print("=" * 60)
for attr_type in ATTR_TYPES:
    s = stats[attr_type]
    print(f"{attr_type:20s}: {s['positive_count']:4d}/{s['total_count']:4d} ({s['positive_rate']:.2%})")

## 4. Visualize Label Distribution

In [ ]:
# Bar plot of positive rates
attr_names = ATTR_TYPES
positive_rates = [stats[attr]['positive_rate'] for attr in ATTR_TYPES]

plt.figure(figsize=(12, 6))
plt.bar(range(len(attr_names)), positive_rates, color='steelblue')
plt.xticks(range(len(attr_names)), attr_names, rotation=45, ha='right')
plt.ylabel('Positive Rate')
plt.title('Attribute Positive Rates in Dataset')
plt.axhline(y=0.5, color='r', linestyle='--', label='50% threshold')
plt.legend()
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Visualize Sample Images with Attributes

In [ ]:
# Visualize a few samples
def visualize_sample(image_id, labels):
    """Visualize image with its attribute masks"""
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    
    # Load and display original image
    img_path = os.path.join(IMAGE_FOLDER, f"{image_id}.jpg")
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    axes[0, 0].imshow(img)
    axes[0, 0].set_title(f"Image: {image_id}")
    axes[0, 0].axis('off')
    
    # Display attribute masks
    for idx, attr_type in enumerate(ATTR_TYPES):
        row = (idx + 1) // 3
        col = (idx + 1) % 3
        
        mask_path = os.path.join(GT_FOLDER, f"{image_id}_attribute_{attr_type}.png")
        
        if os.path.exists(mask_path):
            mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
            axes[row, col].imshow(mask, cmap='gray')
        else:
            axes[row, col].text(0.5, 0.5, 'Not Found', 
                               ha='center', va='center', transform=axes[row, col].transAxes)
        
        label_val = labels[idx]
        title = f"{attr_type}\nLabel: {int(label_val)}"
        axes[row, col].set_title(title)
        axes[row, col].axis('off')
    
    plt.tight_layout()
    plt.show()

# Show first 3 samples
for i in range(min(3, len(image_ids))):
    visualize_sample(image_ids[i], labels_matrix[i])

## 6. Analyze Label Co-occurrence

In [ ]:
# Compute co-occurrence matrix
n_attrs = len(ATTR_TYPES)
cooccur = np.zeros((n_attrs, n_attrs))

for i in range(n_attrs):
    for j in range(n_attrs):
        cooccur[i, j] = np.sum((labels_matrix[:, i] == 1) & (labels_matrix[:, j] == 1))

# Visualize co-occurrence
plt.figure(figsize=(10, 8))
plt.imshow(cooccur, cmap='Blues', aspect='auto')
plt.colorbar(label='Co-occurrence Count')
plt.xticks(range(n_attrs), ATTR_TYPES, rotation=45, ha='right')
plt.yticks(range(n_attrs), ATTR_TYPES)
plt.title('Attribute Co-occurrence Matrix')

# Add text annotations
for i in range(n_attrs):
    for j in range(n_attrs):
        plt.text(j, i, int(cooccur[i, j]), 
                ha='center', va='center', color='white' if cooccur[i, j] > cooccur.max()/2 else 'black')

plt.tight_layout()
plt.show()

## 7. Summary

Key findings:
- Dataset processed into multilabel format (5 attributes)
- Feature threshold = 0.1 (10% of pixels)
- Each image has a 5-dimensional binary label vector
- Ready for multilabel classification training